# Air Hockey — SAC → Diffusion Policy → DPPO

Full training pipeline for the [akshan-main/airhockey-rl-sac-dppo](https://github.com/akshan-main/airhockey-rl-sac-dppo) project.

**Before you start:**
1. `Runtime` → `Change runtime type` → **T4 GPU**
2. Click `Connect` in the top right and wait for the RAM/disk meter
3. Run the cells one at a time in order

**Estimated total wall clock:** ~5-7 hours
- Cell 3 (SAC): 1-2h
- Cell 4 (collect): ~10 min
- Cell 5 (BC): 30-45 min
- Cell 6 (DPPO): 3-4h
- Everything else: <5 min total

Free Colab sessions last ~12h, so it fits in one sitting as long as you keep the tab open and click the notebook every ~30-45 min to show activity.

## Cell 1 — Clone + install + GPU check

~2-3 min. Verifies you're on a T4.

In [ ]:
!git clone https://github.com/akshan-main/airhockey-rl-sac-dppo.git
%cd airhockey-rl-sac-dppo/airhockey-rl
!pip install -q -e ".[storage,app,hub]"

import torch
print(f"torch {torch.__version__}, cuda: {torch.cuda.is_available()}")
assert torch.cuda.is_available(), "Switch runtime to T4 GPU first (Runtime → Change runtime type → T4 GPU)"

## Cell 2 — Import sanity check

~5 sec. Verifies the env instantiates correctly and the action/obs spaces are right.

In [ ]:
from airhockey.env import AirHockeyEnv
from airhockey.sac import SACAgent, SACConfig
from airhockey.policy import DiffusionPolicyConfig, UNet1D

env = AirHockeyEnv(seed=0)
obs, _ = env.reset()
print(f"obs shape: {obs.shape}, dtype: {obs.dtype}")
print(f"action_space: {env.action_space}")
print(f"obs range: [{obs.min():.3f}, {obs.max():.3f}]")

# Quick forward pass through the diffusion policy to make sure torch + the
# model construction work on this Colab machine.
cfg = DiffusionPolicyConfig()
model = UNet1D(cfg).cuda()
dummy_a = torch.randn(2, cfg.horizon, cfg.act_dim, device='cuda')
dummy_t = torch.zeros(2, dtype=torch.long, device='cuda')
dummy_o = torch.randn(2, cfg.obs_dim, device='cuda')
out = model(dummy_a, dummy_t, dummy_o)
print(f"diffusion policy forward ok: output {tuple(out.shape)}, params {sum(p.numel() for p in model.parameters())/1e6:.2f}M")

## Cell 3 — Stage 1: train the SAC expert via self-play (~1-2h)

**This is the long one.** Kick it off and leave it running in the background while you work on other things in another tab.

**What to watch in the tqdm bar:**
- `ret` should climb from roughly -1 to +3..+5 over 1M steps
- `len` (episode length) should first shrink (agent learns to score fast) then stabilize around 200-400

**If `ret` is still flat or negative at step 500K**, stop and tell me — something's wrong (check the `airhockey-rl-doctor` Claude Skill for the Stage 1 decision tree).

**Self-play schedule:**
- First 50K steps: no-op opponent (top paddle sits still) so the agent learns basics
- Then: frozen snapshot of the current actor, refreshed every 50K steps

In [ ]:
!python -m airhockey.train_sac \
    --total-steps 1000000 \
    --opponent-warmup-steps 50000 \
    --opponent-refresh-steps 50000 \
    --out ckpt/sac_expert.pt

## Cell 4 — Stage 2a: collect demonstrations from the SAC expert (~10-15 min)

Rolls out two copies of the trained SAC agent against each other for 5000 episodes and saves every (obs, action) step from the bottom paddle's perspective. This becomes the BC training set.

Expected output file size: **50-150 MB** depending on average episode length.

In [ ]:
!python -m airhockey.collect \
    --expert ckpt/sac_expert.pt \
    --episodes 5000 \
    --out data/demos.npz \
    --device cuda

!ls -lh data/demos.npz

import numpy as np
d = np.load('data/demos.npz')
print(f"obs: {d['obs'].shape}, act: {d['act'].shape}, episodes: {int(d['episode'].max())+1}")
print(f"action range: [{d['act'].min():.3f}, {d['act'].max():.3f}]")

## Cell 5 — Stage 2b: train the Diffusion Policy via BC (~30-45 min)

Trains a 1D UNet noise predictor with the standard DDPM ε-prediction loss on the SAC demonstrations.

**Expected loss curve:**
- Starts around 1.0
- Drops fast to ~0.1 in the first 20 epochs
- Ends near 0.01-0.03 by epoch 200

**If loss is stuck >0.1 at epoch 50**, the dataset probably isn't clean — check the `act` array stats from Cell 4. If the actions are clumped at [-1,1] saturation, Stage 1's SAC was probably running too hot.

In [ ]:
!python -m airhockey.train_bc \
    --data data/demos.npz \
    --epochs 200 \
    --batch-size 512 \
    --lr 1e-4 \
    --horizon 8 \
    --out ckpt/bc.pt

## Cell 6 — Stage 3: DPPO fine-tune (~3-4h)

**The longest stage.** PPO over the diffusion sampling chain, with KL-to-BC regularization against the frozen BC checkpoint. The opponent is the frozen SAC expert from Stage 1 — so the Diffusion Policy is learning to *beat* SAC while staying close to its BC-distilled start.

**What to watch in the tqdm bar:**
- `ret`: should climb over the course of training
- `win`: should climb from ~0.5 (parity with SAC) upward
- `kl`: should stay under 0.05 — if it spikes, lower `--actor-lr` to 1e-5
- `clip`: should stay under 0.15 — if >0.3, raise `--sigma` to 0.2

**If it's clearly failing (win rate dropping below BC's starting point for an extended time),** stop it (interrupt the cell) and use `ckpt/bc.pt` as your final model. The BC checkpoint alone is still a legitimate deployable policy.

In [ ]:
!python -m airhockey.train_dppo \
    --init ckpt/bc.pt \
    --opponent ckpt/sac_expert.pt \
    --total-steps 2000000 \
    --n-envs 16 \
    --rollout-steps 128 \
    --actor-lr 3e-5 \
    --critic-lr 3e-4 \
    --bc-kl-coef 0.05 \
    --out ckpt/dppo.pt

## Cell 7 — Evaluate all checkpoints head-to-head vs the SAC expert

Runs 200 episodes each. The headline number is DPPO's win rate vs SAC — anything ≥ 0.55 is a real improvement over BC.

**Write down the numbers — they go into `MODEL_CARD.md` later.**

In [ ]:
# BC diffusion policy against the SAC expert
print("=== BC vs SAC ===")
!python -m airhockey.eval --ckpt ckpt/bc.pt --opponent ckpt/sac_expert.pt --episodes 200

# DPPO diffusion policy against the SAC expert
print("\n=== DPPO vs SAC ===")
!python -m airhockey.eval --ckpt ckpt/dppo.pt --opponent ckpt/sac_expert.pt --episodes 200

## Cell 8 — Export to ONNX for browser inference

Produces `onnx/policy.onnx` (the noise predictor) and `onnx/policy.json` (DDIM schedule metadata). The browser's JS DDIM sampler needs both files.

In [ ]:
!python -m airhockey.export_onnx --ckpt ckpt/dppo.pt --out onnx
!ls -lh onnx/

## Cell 9 — Push to Hugging Face Hub

**Before running this cell you need to add your HF token as a Colab secret.**

1. Click the 🔑 key icon in the left sidebar
2. Click `Add new secret`
3. Name: `HF_TOKEN`
4. Value: your write-scope token from https://huggingface.co/settings/tokens
5. Toggle `Notebook access` ON

You also need to have created the target HF model repo already:
- https://huggingface.co/new → name `airhockey-dppo` → Model

**If you don't want to publish yet**, skip this cell and go straight to Cell 10 to download the artifacts locally.

In [ ]:
import os
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["HF_REPO_ID"] = "akshan-main/airhockey-dppo"

!python scripts/upload_to_hub.py

## Cell 10 — Download the trained artifacts to your local machine

Move these files into your local `airhockey-rl/ckpt/` and `airhockey-rl/onnx/` directories. Your FastAPI backend will then serve `onnx/policy.onnx` to the browser.

In [ ]:
from google.colab import files

files.download("ckpt/sac_expert.pt")
files.download("ckpt/bc.pt")
files.download("ckpt/dppo.pt")
files.download("onnx/policy.onnx")
files.download("onnx/policy.json")

## Cell 11 — (Optional) Publish the demonstration dataset as a HF Dataset

In [ ]:
# Create the dataset repo on HF first (https://huggingface.co/new-dataset)
# Name: airhockey-demos

os.environ["HF_DATASET_REPO"] = "akshan-main/airhockey-demos"
!python scripts/upload_demos_to_dataset.py

## Cell 12 — (Optional) Build + publish the Minari dataset

In [ ]:
!python scripts/build_minari_dataset.py --data data/demos.npz --id airhockey-sac-v0
# Uncomment to publicly upload to Minari (requires a Farama / HF account):
# !minari upload airhockey-sac-v0

## Done — what to do next

1. **Deploy the backend** — create an HF Space (Docker SDK), push this repo to it, and set the required secrets. Walkthrough in `scripts/setup_hf_space.md`.
2. **Wire the browser** — open `play.html`, confirm `window.AIRHOCKEY_BACKEND` points at your Space URL, and start playing.
3. **Watch the online retrain loop** — every 6h the GitHub Actions cron pulls buffered human gameplay from HF Buckets, runs a BC-on-winners update, and pushes the new model back to HF Hub. The backend hot-reloads within 60s and the browser within another 10s.

If any training stage goes sideways, load the `airhockey-rl-doctor` Claude Skill and paste your training log — it encodes the failure modes and fixes for each stage.

**Final eval numbers from this training run** (fill in after you see the Cell 7 output):

| Stage       | Win rate vs SAC | Mean return |
|-------------|-----------------|-------------|
| BC          |                 |             |
| DPPO        |                 |             |